# GNN Proof of Concept - J03WMX Episode 1

Notebook ini menyambungkan Episode 1 J03WMX ke pipeline graph dan GNN.

Metodologi yang dipertahankan:
- turnover adalah active possession change;
- potential regain adalah penguasaan kembali oleh losing team dalam 5 detik;
- Episode 1 adalah potential regain pertama;
- window PoC dimulai 2 detik sebelum loss dan berakhir saat regain;
- graph menggunakan kNN dengan k=4 seperti PoC sebelumnya.

Penyesuaian untuk konsistensi lintas pertandingan: ID tim absolut diganti dengan peran relatif `is_losing_team` dan `is_opponent`. Penyesuaian ini tidak mengubah deteksi turnover/regain.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data" / "sportec_idsse").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data" / "sportec_idsse"
MATCH_ID = "J03WMX"
K_NEIGHBORS = 4

tracking_path = DATA_DIR / f"{MATCH_ID}_tracking.csv"
events_path = DATA_DIR / f"{MATCH_ID}_events.csv"

if not tracking_path.exists() or not events_path.exists():
    raise FileNotFoundError(
        f"Dataset J03WMX tidak ditemukan di {DATA_DIR}"
    )

print("Data directory:", DATA_DIR)

Data directory: c:\VAEP-Track\data\sportec_idsse


In [2]:
tracking = pd.read_csv(tracking_path)
events = pd.read_csv(events_path)

tracking["timestamp"] = pd.to_timedelta(
    tracking["timestamp"],
    errors="coerce"
)
tracking = tracking.sort_values(
    ["period_id", "timestamp", "frame_id"]
).reset_index(drop=True).copy()

tracking["prev_owning_team"] = (
    tracking.groupby("period_id")["ball_owning_team_id"]
    .shift(1)
)
tracking["team_changed"] = (
    tracking["ball_owning_team_id"].notna()
    & tracking["prev_owning_team"].notna()
    & (
        tracking["ball_owning_team_id"]
        != tracking["prev_owning_team"]
    )
)

candidate_changes = tracking[
    tracking["team_changed"]
    & (tracking["ball_state"] == "alive")
].copy()
candidate_changes = candidate_changes.sort_values(
    ["period_id", "timestamp", "frame_id"]
).reset_index(drop=True)

regain_events = []
for idx, row in candidate_changes.iterrows():
    loss_time = pd.Timedelta(row["timestamp"])
    losing_team = row["prev_owning_team"]
    period_id = row["period_id"]

    future = candidate_changes[
        (candidate_changes["period_id"] == period_id)
        & (candidate_changes.index > idx)
        & (candidate_changes["timestamp"] > loss_time)
        & (
            candidate_changes["timestamp"]
            <= loss_time + pd.to_timedelta(5, unit="s")
        )
        & (
            candidate_changes["ball_owning_team_id"]
            == losing_team
        )
    ]

    if future.empty:
        continue

    regain = future.iloc[0]
    regain_time = pd.Timedelta(regain["timestamp"])
    regain_events.append({
        "match_id": MATCH_ID,
        "period_id": int(period_id),
        "loss_timestamp": loss_time,
        "regain_timestamp": regain_time,
        "loss_frame": int(row["frame_id"]),
        "regain_frame": int(regain["frame_id"]),
        "losing_team": losing_team,
        "opponent_team": row["ball_owning_team_id"],
        "regain_gap_seconds": (
            regain_time - loss_time
        ).total_seconds()
    })

regain_events_df = pd.DataFrame(regain_events)
episode = regain_events_df.iloc[0].copy()

t0 = pd.Timedelta(episode["loss_timestamp"])
t1 = pd.Timedelta(episode["regain_timestamp"])
period_id = episode["period_id"]
window_start = t0 - pd.to_timedelta(2, unit="s")
window_end = t1

episode_window = tracking[
    (tracking["period_id"] == period_id)
    & (tracking["timestamp"] >= window_start)
    & (tracking["timestamp"] <= window_end)
].copy()

print("Active possession changes:", len(candidate_changes))
print("Potential regain <= 5 detik:", len(regain_events_df))
print("Episode 1 loss:", t0)
print("Episode 1 regain:", t1)
print("Regain gap:", episode["regain_gap_seconds"])
print("Window frames:", len(episode_window))

Active possession changes: 408
Potential regain <= 5 detik: 211
Episode 1 loss: 0 days 00:01:22.040000
Episode 1 regain: 0 days 00:01:25.320000
Regain gap: 3.2800000000000002
Window frames: 133


In [3]:
player_team_map = (
    events[["player_id", "team_id"]]
    .dropna()
    .drop_duplicates()
    .groupby("player_id")["team_id"]
    .first()
    .to_dict()
)

loss_row = tracking[
    (tracking["period_id"] == period_id)
    & (tracking["frame_id"] == episode["loss_frame"])
].iloc[0]

episode_teams = {
    episode["losing_team"],
    episode["opponent_team"]
}
player_ids_tracking = sorted({
    column[:-2]
    for column in tracking.columns
    if column.startswith("DFL-OBJ-") and column.endswith("_x")
})

active_player_ids = []
for player_id in player_ids_tracking:
    x_col = f"{player_id}_x"
    y_col = f"{player_id}_y"
    team_id = player_team_map.get(player_id)

    if (
        team_id in episode_teams
        and pd.notna(loss_row.get(x_col, np.nan))
        and pd.notna(loss_row.get(y_col, np.nan))
    ):
        active_player_ids.append(player_id)

node_order = active_player_ids + ["BALL"]

print("Active players at loss:", len(active_player_ids))
print("Fixed node count:", len(node_order))
print("Losing-team players:", sum(
    player_team_map[player_id] == episode["losing_team"]
    for player_id in active_player_ids
))
print("Opponent players:", sum(
    player_team_map[player_id] == episode["opponent_team"]
    for player_id in active_player_ids
))

if len(active_player_ids) != 22:
    raise ValueError(
        f"Episode 1 diharapkan memiliki 22 pemain, ditemukan {len(active_player_ids)}"
    )

Active players at loss: 22
Fixed node count: 23
Losing-team players: 11
Opponent players: 11


In [4]:
def build_frame_graph(
    frame,
    node_order,
    player_team_map,
    losing_team,
    opponent_team,
    t0,
    k=4
):
    nodes = []

    for node_id in node_order:
        if node_id == "BALL":
            x = frame["ball_x"]
            y = frame["ball_y"]
            speed = frame.get("ball_speed", np.nan)
            team_id = None
            is_ball = 1.0
        else:
            x = frame.get(f"{node_id}_x", np.nan)
            y = frame.get(f"{node_id}_y", np.nan)
            speed = frame.get(f"{node_id}_s", np.nan)
            team_id = player_team_map.get(node_id)
            is_ball = 0.0

        is_valid = pd.notna(x) and pd.notna(y)
        nodes.append({
            "node_id": node_id,
            "x": float(x) if is_valid else 0.0,
            "y": float(y) if is_valid else 0.0,
            "speed": (
                float(speed) if pd.notna(speed) else 0.0
            ),
            "is_ball": is_ball,
            "is_losing_team": float(
                team_id == losing_team
            ),
            "is_opponent": float(
                team_id == opponent_team
            ),
            "is_valid": bool(is_valid)
        })

    nodes_df = pd.DataFrame(nodes)
    valid_indices = nodes_df.index[nodes_df["is_valid"]].to_numpy()
    positions = nodes_df.loc[valid_indices, ["x", "y"]].to_numpy()
    edges = []

    for local_source, source_idx in enumerate(valid_indices):
        distances = np.linalg.norm(
            positions - positions[local_source],
            axis=1
        )
        neighbor_order = np.argsort(distances)
        neighbor_order = neighbor_order[neighbor_order != local_source][:k]

        for local_target in neighbor_order:
            target_idx = valid_indices[local_target]
            edges.append({
                "source_idx": int(source_idx),
                "target_idx": int(target_idx),
                "distance": float(distances[local_target])
            })

    edges_df = pd.DataFrame(edges)
    relative_time = (frame["timestamp"] - t0).total_seconds()

    return {
        "frame_id": int(frame["frame_id"]),
        "relative_time": float(relative_time),
        "nodes": nodes_df,
        "edges": edges_df
    }


graph_sequence = [
    build_frame_graph(
        frame=frame,
        node_order=node_order,
        player_team_map=player_team_map,
        losing_team=episode["losing_team"],
        opponent_team=episode["opponent_team"],
        t0=t0,
        k=K_NEIGHBORS
    )
    for _, frame in episode_window.iterrows()
]

node_counts = [len(item["nodes"]) for item in graph_sequence]
edge_counts = [len(item["edges"]) for item in graph_sequence]
invalid_counts = [
    int((~item["nodes"]["is_valid"]).sum())
    for item in graph_sequence
]

print("Graph frames:", len(graph_sequence))
print("Node counts:", min(node_counts), "to", max(node_counts))
print("Edge counts:", min(edge_counts), "to", max(edge_counts))
print("Maximum invalid nodes in one frame:", max(invalid_counts))
print("First frame:", graph_sequence[0]["frame_id"])
print("Last frame:", graph_sequence[-1]["frame_id"])

Graph frames: 133
Node counts: 23 to 23
Edge counts: 92 to 92
Maximum invalid nodes in one frame: 0
First frame: 12001
Last frame: 12133


In [5]:
import torch
from torch_geometric.data import Data

NODE_FEATURES = [
    "x",
    "y",
    "speed",
    "is_ball",
    "is_losing_team",
    "is_opponent"
]


def graph_to_pyg(graph_item):
    nodes_df = graph_item["nodes"]
    edges_df = graph_item["edges"]

    x = torch.tensor(
        nodes_df[NODE_FEATURES].to_numpy(),
        dtype=torch.float32
    )
    pos = torch.tensor(
        nodes_df[["x", "y"]].to_numpy(),
        dtype=torch.float32
    )
    edge_index = torch.tensor(
        edges_df[["source_idx", "target_idx"]].to_numpy(),
        dtype=torch.long
    ).t().contiguous()
    edge_attr = torch.tensor(
        edges_df[["distance"]].to_numpy(),
        dtype=torch.float32
    )
    node_mask = torch.tensor(
        nodes_df["is_valid"].to_numpy(),
        dtype=torch.bool
    )

    return Data(
        x=x,
        pos=pos,
        edge_index=edge_index,
        edge_attr=edge_attr,
        node_mask=node_mask,
        frame_id=graph_item["frame_id"],
        relative_time=graph_item["relative_time"]
    )


pyg_sequence = [
    graph_to_pyg(item)
    for item in graph_sequence
]

print("PyG graph count:", len(pyg_sequence))
print("First graph:", pyg_sequence[0])
print("Last graph:", pyg_sequence[-1])

PyG graph count: 133
First graph: Data(x=[23, 6], edge_index=[2, 92], edge_attr=[92, 1], pos=[23, 2], node_mask=[23], frame_id=12001, relative_time=-2.0)
Last graph: Data(x=[23, 6], edge_index=[2, 92], edge_attr=[92, 1], pos=[23, 2], node_mask=[23], frame_id=12133, relative_time=3.2800000000000002)


In [6]:
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv


class GNNPoC(nn.Module):
    def __init__(self, in_channels=6, hidden_channels=32, output_channels=2):
        super().__init__()
        self.gcn1 = GCNConv(in_channels, hidden_channels)
        self.gcn2 = GCNConv(hidden_channels, hidden_channels)
        self.classifier = nn.Linear(hidden_channels, output_channels)

    def forward(self, graphs):
        graph_embeddings = []

        for graph in graphs:
            h = F.relu(self.gcn1(graph.x, graph.edge_index))
            h = F.relu(self.gcn2(h, graph.edge_index))
            valid_h = h[graph.node_mask]
            graph_embeddings.append(valid_h.mean(dim=0))

        temporal_embedding = torch.stack(graph_embeddings)
        episode_embedding = temporal_embedding.mean(dim=0)
        logits = self.classifier(episode_embedding)
        return logits, episode_embedding


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = GNNPoC().to(device)
pyg_sequence_device = [graph.to(device) for graph in pyg_sequence]

model.eval()
with torch.no_grad():
    logits, episode_embedding = model(pyg_sequence_device)

print("Device:", device)
print("Logit shape:", logits.shape)
print("Embedding shape:", episode_embedding.shape)
print("Untrained logits:", logits)
print("Catatan: logits belum terlatih dan bukan hasil prediksi.")

Device: cuda
Logit shape: torch.Size([2])
Embedding shape: torch.Size([32])
Untrained logits: tensor([-0.7902, -0.2162], device='cuda:0')
Catatan: logits belum terlatih dan bukan hasil prediksi.
